# RNNs and Sequence Models

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/03-deep-learning/04_rnns_sequence_models.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Process sequential data with RNNs, LSTMs, and GRUs — understand vanishing gradients, sequence-to-sequence patterns, and when transformers replaced them.

**Prerequisites:** CNNs (notebook 03)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy matplotlib torch


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. Why Sequences Need Special Architectures

In [ ]:
# Demonstrate that order matters in sequences
sentence1 = "The cat sat on the mat"
sentence2 = "mat the on sat cat The"

print(f"Original:  '{sentence1}'")
print(f"Shuffled:  '{sentence2}'")
print(f"Same words: {sorted(sentence1.split()) == sorted(sentence2.split())}")
print(f"Same meaning: No! Order carries information.")
print(f"\nA feedforward network sees features independently.")
print(f"A recurrent network processes tokens one by one, maintaining state.")

## 2. Vanilla RNN

In [ ]:
# Manual RNN cell to show the math
input_size, hidden_size = 4, 8
rnn = nn.RNN(input_size, hidden_size, batch_first=True)

seq_len = 10
x = torch.randn(1, seq_len, input_size)
output, h_n = rnn(x)

print(f"Input:  {x.shape}  (batch, seq_len, features)")
print(f"Output: {output.shape}  (batch, seq_len, hidden)")
print(f"h_n:    {h_n.shape}  (layers, batch, hidden)")
print(f"\noutput[:, -1, :] == h_n:  {torch.allclose(output[:, -1, :], h_n[0])}")

## 3. LSTM and GRU — Solving Vanishing Gradients

In [ ]:
lstm = nn.LSTM(input_size=4, hidden_size=8, batch_first=True)
gru = nn.GRU(input_size=4, hidden_size=8, batch_first=True)

x = torch.randn(1, 10, 4)

lstm_out, (h_lstm, c_lstm) = lstm(x)
gru_out, h_gru = gru(x)

print("LSTM:")
print(f"  Output: {lstm_out.shape}")
print(f"  Hidden: {h_lstm.shape}, Cell: {c_lstm.shape}")
print(f"  Params: {sum(p.numel() for p in lstm.parameters()):,}")

print(f"\nGRU:")
print(f"  Output: {gru_out.shape}")
print(f"  Hidden: {h_gru.shape}")
print(f"  Params: {sum(p.numel() for p in gru.parameters()):,}")
print(f"\nGRU has ~25% fewer parameters (no cell state)")

## 4. Sequence Classification (Sentiment Analysis)

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        output, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

# Demo with random data
vocab_size, embed_dim, hidden_dim = 5000, 64, 128
model = SentimentRNN(vocab_size, embed_dim, hidden_dim, output_dim=1)

fake_input = torch.randint(0, vocab_size, (4, 20))
output = model(fake_input)
print(f"Input: {fake_input.shape} (batch=4, seq_len=20)")
print(f"Output: {output.shape} (batch=4, 1 sentiment score)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Time Series Prediction

In [ ]:
# Generate sine wave data
t = np.linspace(0, 100, 1000)
data = np.sin(t) + np.random.normal(0, 0.05, len(t))

# Create sequences
def create_sequences(data, seq_len):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        xs.append(data[i:i+seq_len])
        ys.append(data[i+seq_len])
    return torch.FloatTensor(np.array(xs)).unsqueeze(-1), torch.FloatTensor(np.array(ys))

seq_len = 50
X_seq, y_seq = create_sequences(data, seq_len)
split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

class TimeSeriesLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()

model = TimeSeriesLSTM()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(30):
    model.train()
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} | Loss: {loss.item():.6f}")

model.eval()
with torch.no_grad():
    preds = model(X_test).numpy()

plt.figure(figsize=(12, 4))
plt.plot(y_test.numpy(), label='Actual', alpha=0.7)
plt.plot(preds, label='Predicted', alpha=0.7)
plt.title('LSTM Sine Wave Prediction')
plt.legend()
plt.show()

## 6. From RNNs to Attention (Motivation for Transformers)

In [ ]:
# Show the bottleneck problem
print("RNN Bottleneck:")
print("  Input: 'The cat that the dog chased ran away'")
print("  RNN must compress the ENTIRE sentence into a fixed-size vector")
print("  before the decoder can use it.\n")

# Demonstrate with hidden states
rnn = nn.LSTM(8, 16, batch_first=True)
long_seq = torch.randn(1, 100, 8)
short_seq = torch.randn(1, 5, 8)

_, (h_long, _) = rnn(long_seq)
_, (h_short, _) = rnn(short_seq)

print(f"100-token sequence → hidden: {h_long.shape} (same size!)")
print(f"  5-token sequence → hidden: {h_short.shape} (same size!)")
print(f"\nInformation bottleneck: a 100-token sequence is compressed")
print(f"into the same {h_long.shape[-1]}-dim vector as a 5-token one.")
print(f"\nAttention solves this by letting the decoder look at ALL")
print(f"encoder hidden states, not just the final one.")
print(f"\n→ This insight led to the Transformer (next section: 04-llm-and-transformers)")

## Try It Yourself

1. Train an LSTM to predict a sine wave. Plot the predicted vs actual values.
2. Build a character-level RNN that generates text after training on a small text corpus.
3. Compare RNN vs LSTM vs GRU on a sequence classification task. Plot accuracy curves for all three.

In [ ]:
# Your code here